# TARF Risk Profile: Target Redemption Forward

A TARF accumulates profit on each fixing where spot > strike, but terminates when cumulative profit hits the target. This creates an **asymmetric payoff**: upside is capped, downside is uncapped.

**What we build:**
1. TARF pricing at different spot levels — the asymmetric payoff
2. Target level sensitivity — how the cap drives early termination
3. Mark-to-market path dependency — what happens mid-life
4. Vol sensitivity — why TARFs are short vol on the upside
5. Comparison: TARF vs vanilla forward

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "python"))

import numpy as np
from pricebook.fx_structured import fx_tarf_price
from pricebook.viz import configure_theme
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

# Base case: EUR corporate hedging USD receivables
# Long USD via TARF (buy USD / sell EUR)
SPOT = 1.0900     # EURUSD
STRIKE = 1.0900   # ATM
TARGET = 0.10     # target profit = 10 cents
R_DOM = 0.037     # EUR rate
R_FOR = 0.053     # USD rate
VOL = 0.08        # EURUSD vol
T = 1.0           # 1 year
N_OBS = 12        # monthly fixings
NOTIONAL = 1_000_000
N_PATHS = 20_000

base = fx_tarf_price(SPOT, STRIKE, TARGET, R_DOM, R_FOR, VOL, T,
                     N_OBS, NOTIONAL, n_paths=N_PATHS)

print(f"TARF: EURUSD, strike={STRIKE}, target={TARGET}, {N_OBS} monthly fixings")
print(f"Notional: {NOTIONAL:,.0f} per fixing")
print(f"\nPrice:              {base.price:,.0f}")
print(f"Expected profit:    {base.expected_profit:,.0f}")
print(f"P(target hit):      {base.prob_target_hit*100:.1f}%")
print(f"Mean termination:   {base.mean_termination_time:.1f} months")

## 1. Payoff asymmetry — TARF vs forward at different spots

In [ ]:
spots = np.linspace(1.02, 1.16, 15)
tarf_prices = []
fwd_pnl = []

print(f"{'Spot':>7}  {'TARF PV':>12}  {'Fwd PnL':>12}  {'P(target)':>10}  {'Mean Term':>10}")
print(f"{'─'*7}  {'─'*12}  {'─'*12}  {'─'*10}  {'─'*10}")

for s in spots:
    r = fx_tarf_price(s, STRIKE, TARGET, R_DOM, R_FOR, VOL, T,
                      N_OBS, NOTIONAL, n_paths=N_PATHS)
    tarf_prices.append(r.price)
    
    # Vanilla forward PnL for comparison (12 monthly forwards)
    import math
    fwd = s * math.exp((R_DOM - R_FOR) * T)
    fwd_total = NOTIONAL * N_OBS * (fwd - STRIKE)
    fwd_pnl.append(fwd_total)
    
    print(f"{s:>7.4f}  {r.price:>+12,.0f}  {fwd_total:>+12,.0f}  "
          f"{r.prob_target_hit*100:>9.1f}%  {r.mean_termination_time*12:>9.1f}m")

# Plot
with apply_theme():
    fig, ax = create_figure(1)
    ax.plot(spots, [p/1000 for p in tarf_prices], 'o-', linewidth=2, label='TARF')
    ax.plot(spots, [p/1000 for p in fwd_pnl], 's--', linewidth=2, label='Vanilla Forward')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(STRIKE, color='tab:gray', linewidth=0.5, linestyle=':')
    ax.set_xlabel('EURUSD Spot')
    ax.set_ylabel('PV ($000s)')
    ax.set_title('TARF vs Vanilla Forward: Asymmetric Payoff')
    ax.legend(fontsize=9)
    fig.tight_layout()

print("\nKey: upside capped by target, downside uncapped — negative convexity.")

## 2. Target level sensitivity — how the cap changes behaviour

In [ ]:
targets = [0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.50]
prices_by_target = []
probs_by_target = []
terms_by_target = []

print(f"{'Target':>7}  {'Price':>12}  {'P(hit)':>8}  {'Mean Term':>10}  {'Exp Profit':>12}")
print(f"{'─'*7}  {'─'*12}  {'─'*8}  {'─'*10}  {'─'*12}")

for tgt in targets:
    r = fx_tarf_price(SPOT, STRIKE, tgt, R_DOM, R_FOR, VOL, T,
                      N_OBS, NOTIONAL, n_paths=N_PATHS)
    prices_by_target.append(r.price)
    probs_by_target.append(r.prob_target_hit * 100)
    terms_by_target.append(r.mean_termination_time * 12)
    print(f"{tgt:>7.2f}  {r.price:>+12,.0f}  {r.prob_target_hit*100:>7.1f}%  "
          f"{r.mean_termination_time*12:>9.1f}m  {r.expected_profit:>+12,.0f}")

# Plot
with apply_theme():
    fig, (ax1, ax2) = create_figure(2)
    
    ax1.plot(targets, [p/1000 for p in prices_by_target], 'o-', linewidth=2)
    ax1.set_xlabel('Target Level')
    ax1.set_ylabel('TARF Price ($000s)')
    ax1.set_title('Price vs Target: Higher Target = More Upside')
    
    ax2.plot(targets, probs_by_target, 'o-', linewidth=2, label='P(target hit)', color='tab:red')
    ax2_twin = ax2.twinx()
    ax2_twin.plot(targets, terms_by_target, 's--', linewidth=2, label='Mean termination', color='tab:blue')
    ax2.set_xlabel('Target Level')
    ax2.set_ylabel('P(target hit) %', color='tab:red')
    ax2_twin.set_ylabel('Mean Term (months)', color='tab:blue')
    ax2.set_title('Early Termination vs Target Level')
    
    fig.tight_layout()

## 3. Vol sensitivity — TARFs are short vol on the upside

In [ ]:
vols = np.arange(0.04, 0.16, 0.01)
prices_by_vol = []

print(f"{'Vol':>5}  {'Price':>12}  {'P(hit)':>8}  {'Exp Profit':>12}")
print(f"{'─'*5}  {'─'*12}  {'─'*8}  {'─'*12}")

for v in vols:
    r = fx_tarf_price(SPOT, STRIKE, TARGET, R_DOM, R_FOR, v, T,
                      N_OBS, NOTIONAL, n_paths=N_PATHS)
    prices_by_vol.append(r.price)
    print(f"{v*100:>4.0f}%  {r.price:>+12,.0f}  {r.prob_target_hit*100:>7.1f}%  {r.expected_profit:>+12,.0f}")

# Plot
with apply_theme():
    fig, ax = create_figure(1)
    ax.plot(vols * 100, [p/1000 for p in prices_by_vol], 'o-', linewidth=2)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('FX Vol (%)')
    ax.set_ylabel('TARF Price ($000s)')
    ax.set_title('TARF Price vs Volatility')
    fig.tight_layout()

print("\nHigher vol → more downside exposure (uncapped) but target hit faster (capped upside).")
print("Net effect depends on the moneyness — ATM TARFs are typically short vol.")

## 4. Strike sensitivity — ITM vs OTM

In [ ]:
strikes = np.linspace(1.05, 1.13, 9)
prices_by_strike = []

print(f"{'Strike':>7}  {'Price':>12}  {'P(hit)':>8}  {'Mean Term':>10}")
print(f"{'─'*7}  {'─'*12}  {'─'*8}  {'─'*10}")

for k in strikes:
    r = fx_tarf_price(SPOT, k, TARGET, R_DOM, R_FOR, VOL, T,
                      N_OBS, NOTIONAL, n_paths=N_PATHS)
    prices_by_strike.append(r.price)
    print(f"{k:>7.4f}  {r.price:>+12,.0f}  {r.prob_target_hit*100:>7.1f}%  "
          f"{r.mean_termination_time*12:>9.1f}m")

with apply_theme():
    fig, ax = create_figure(1)
    ax.plot(strikes, [p/1000 for p in prices_by_strike], 'o-', linewidth=2)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(SPOT, color='tab:gray', linewidth=0.5, linestyle=':', label=f'Spot ({SPOT})')
    ax.set_xlabel('Strike')
    ax.set_ylabel('TARF Price ($000s)')
    ax.set_title('TARF Price vs Strike')
    ax.legend(fontsize=9)
    fig.tight_layout()

## Summary

| Feature | TARF | Vanilla Forward |
|---------|------|----------------|
| Upside | **Capped** at target | Unlimited |
| Downside | **Uncapped** | Unlimited |
| Convexity | **Negative** | Zero |
| Vol exposure | **Short vol** (upside capped) | Neutral |
| Path dependency | **Yes** (early termination) | No |
| Cost | Cheaper (the cap is sold) | Fair value |

**Key insights:**
- TARFs are popular because they provide **cheaper hedging** (or zero-cost) vs vanilla forwards
- The trade-off: the corporate **sells the upside** beyond the target in exchange for a better strike
- **Path dependency** makes MTM volatile — two identical final spots can have very different P&Ls
- The **target level** is the key structuring parameter: lower target = more early termination = cheaper
- TARFs are **short gamma** on the upside — the worst scenario is a gradual favourable move that never quite hits the target, followed by a sharp reversal
- Corporates love them; risk managers worry about the uncapped downside